# EXP-025 — Final Test and Holdout (5) from frozen EXP-024 checkpoints

Evaluation only: this notebook does **not train**. It verifies the source
EXP-024 Validation lock, dataset fingerprint and all five checkpoint SHA-256
values before scoring the 685-image Final Test and 120-image Holdout (5).
Both evaluations write to a separate EXP-025 result directory. The Validation
selected deployment seed stays fixed; Test and Holdout do not tune the model.

The pinned source is the completed 024 256px run in the shared Google Drive
project: [EXP-024 result folder](https://drive.google.com/drive/folders/13BEQoXLIdhanQ1zSLRawhAeoBwAQji-M).
The notebook finds it through the mounted project shortcut. Set
`SOURCE_DIR_OVERRIDE` to that mounted folder, or `SOURCE_FULL_ZIP_OVERRIDE` to
its FULL archive if automatic discovery fails. An ANALYSIS archive has no checkpoints.
The Split (11) and Holdout (5) ZIPs must also be mounted, or given through the
archive override paths. Run all cells in order.


In [ ]:

# ============================================================
# 01. IMPORT LIBRARIES
# ============================================================

import os
import base64
import zlib
import sys
import io
import gc
import json
import time
import random
import shutil
import platform
import zipfile
import stat
import tempfile
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    ConfusionMatrixDisplay,
)

from IPython.display import display
import ipywidgets as widgets

MATRIX_CMAP = LinearSegmentedColormap.from_list(
    "readable_blues", ["#ffffff", "#dbeafe", "#93c5fd"]
)

plt.rcParams.update({
    "text.color": "black", "axes.labelcolor": "black",
    "axes.titlecolor": "black", "xtick.color": "black", "ytick.color": "black",
})

print("Imports completed.")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")


In [ ]:

# ============================================================
# 02. ENVIRONMENT INFORMATION
# ============================================================

print(f"Date              : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python            : {sys.version.split()[0]}")
print(f"TensorFlow        : {tf.__version__}")
print(f"Keras             : {keras.__version__ if hasattr(keras, '__version__') else 'tf.keras'}")
RUNTIME = ("Windows" if os.name == "nt" else "Kaggle" if Path("/kaggle/working").is_dir() else "Colab" if Path("/content").is_dir() else platform.system())
print(f"Runtime           : {RUNTIME}")
print(f"OS                : {platform.platform()}")

gpus = tf.config.list_physical_devices("GPU")
print(f"GPU Count         : {len(gpus)}")
if not gpus:
    print("CPU evaluation is available, but may be slow.")

for i, gpu in enumerate(gpus):
    print(f"GPU {i}             : {gpu}")

if gpus:
    strategy = tf.distribute.OneDeviceStrategy(device="/GPU:0")
else:
    strategy = tf.distribute.OneDeviceStrategy(device="/CPU:0")

print(f"Strategy          : {strategy.__class__.__name__}")
print(f"Replicas          : {strategy.num_replicas_in_sync}")


In [ ]:
# EXP-025 configuration: 024 checkpoints are read-only inputs.
SOURCE_DIR_OVERRIDE = None       # e.g. /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-...024-256px-f9fd5d5f
SOURCE_FULL_ZIP_OVERRIDE = None  # mounted Drive path to SOURCE_EXP_ID + "_FULL.zip", or attached Kaggle input
SOURCE_EXP_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-024-256px-f9fd5d5f"
SOURCE_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/13BEQoXLIdhanQ1zSLRawhAeoBwAQji-M"
DATASET_ARCHIVE_OVERRIDE = None
REAL_HOLDOUT_ARCHIVE_OVERRIDE = None
DRIVE_PROJECT_DIR_OVERRIDE = None
DRIVE_RESULTS_ROOT_OVERRIDE = None
FORCE_REMOUNT_DRIVE = False
RUN_FINAL_TEST = True
RUN_REAL_HOLDOUT = True
RUN_FULL_TRAINING = False
RUN_PIPELINE_SMOKE = False
RUN_SMOKE_ONLY = False
VIEW_ONLY = False

DATASET_ARCHIVE_NAME = "CashewData_Split (11).zip"
REAL_HOLDOUT_ARCHIVE_NAME = "RL_Cashew_holdout (5).zip"
EXPECTED_ZIP_SHA256 = "5d60295d26068f5d9f6427dc730a0f662c56c28ad9ee95f8b594faf546861f49"
REAL_HOLDOUT_ARCHIVE_SHA256 = "8D5C3758B5869C3AE19238FA8D78E53AD15BB40544F4C70F632602896D80045F"
REAL_HOLDOUT_EXPECTED_COUNTS = {"anthracnose": 46, "healthy": 13, "leaf_miner": 36, "red_rust": 25}
EXPECTED_CLASSES = ["anthracnose", "healthy", "leaf_miner", "red_rust", "not_cashew_leaf"]
EXPECTED_COUNTS = {
    "anthracnose": {"train": 982, "val": 309, "test": 120, "total": 1411},
    "healthy": {"train": 830, "val": 222, "test": 118, "total": 1170},
    "leaf_miner": {"train": 875, "val": 267, "test": 130, "total": 1272},
    "not_cashew_leaf": {"train": 1102, "val": 314, "test": 156, "total": 1572},
    "red_rust": {"train": 1127, "val": 335, "test": 161, "total": 1623},
}
EXPECTED_TOTALS = {"train": 4916, "val": 1447, "test": 685, "total": 7048}
EXPECTED_REAL_FILENAME_COUNTS = {"train": 238, "val": 54, "test": 28}
REQUIRED_ABSENT_IMAGES = []
STRICT_DATASET_CHECK = True
SEEDS = [42, 123, 2026, 3407, 7777]
IMG_SIZE = 256
MODEL_NAME = "DenseNet121"
EXPERIMENT_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-025-256px"

if RUNTIME == "Windows":
    PROJECT_ROOT = Path(r"D:\Study\KhoaLuanTotNghiep\Resource")
    RESULTS_ROOT = PROJECT_ROOT / "training_results" / "current_dataset"
elif RUNTIME == "Kaggle":
    PROJECT_ROOT = Path("/kaggle/working")
    RESULTS_ROOT = PROJECT_ROOT / "training_results" / "current_dataset"
elif RUNTIME == "Colab":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=FORCE_REMOUNT_DRIVE)
    PROJECT_ROOT = Path("/content")
    drive_root = Path("/content/drive/MyDrive")
    if DRIVE_PROJECT_DIR_OVERRIDE is not None:
        DRIVE_PROJECT_DIR = Path(DRIVE_PROJECT_DIR_OVERRIDE).expanduser()
    else:
        direct = drive_root / "Cashew_Leaf_model_result"
        candidates = [direct] if (direct / "Dataset" / DATASET_ARCHIVE_NAME).is_file() else []
        if not candidates:
            candidates = [p for p in drive_root.iterdir() if p.is_dir() and
                          (p / "Dataset" / DATASET_ARCHIVE_NAME).is_file()]
        if len(candidates) != 1:
            raise FileNotFoundError("Set DRIVE_PROJECT_DIR_OVERRIDE to the project shortcut")
        DRIVE_PROJECT_DIR = candidates[0]
    DRIVE_DATASET_ARCHIVE = DRIVE_PROJECT_DIR / "Dataset" / DATASET_ARCHIVE_NAME
    RESULTS_ROOT = (Path(DRIVE_RESULTS_ROOT_OVERRIDE).expanduser() if
                    DRIVE_RESULTS_ROOT_OVERRIDE is not None else
                    DRIVE_PROJECT_DIR / "training_results" / "current_dataset")
else:
    raise RuntimeError(f"Unsupported runtime: {RUNTIME}")

def _source_candidates():
    roots = [RESULTS_ROOT]
    if RUNTIME == "Kaggle": roots.append(Path("/kaggle/input"))
    found = []
    for root in roots:
        if not root.is_dir(): continue
        found.extend(p.parent for p in root.rglob("validation_selection_lock.json")
                     if p.parent.name == SOURCE_EXP_ID)
    return sorted(set(found))

def _extract_full_archive(archive_path):
    archive_path = Path(archive_path).expanduser()
    if not archive_path.is_file(): raise FileNotFoundError(archive_path)
    if not archive_path.name.endswith("_FULL.zip"):
        raise ValueError("Use EXP-024 FULL.zip; ANALYSIS.zip lacks checkpoints")
    cache = PROJECT_ROOT / "source_024_cache" / archive_path.stem
    marker = cache / ".source_zip_sha256"
    digest = file_sha256(archive_path)
    if cache.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != digest:
            raise RuntimeError(f"Source cache changed: {cache}")
    else:
        cache.mkdir(parents=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                target = (cache / member.filename).resolve()
                if stat.S_ISLNK(member.external_attr >> 16) or (target != cache.resolve() and cache.resolve() not in target.parents):
                    raise ValueError(f"Unsafe source ZIP member: {member.filename}")
            archive.extractall(cache)
        marker.write_text(digest + "\n", encoding="utf-8")
    dirs = [p.parent for p in cache.rglob("validation_selection_lock.json")]
    if len(dirs) != 1: raise RuntimeError(f"Expected one EXP-024 run in {archive_path}; found {dirs}")
    return dirs[0]

if SOURCE_DIR_OVERRIDE is not None and SOURCE_FULL_ZIP_OVERRIDE is not None:
    raise ValueError("Set only one source override")
if SOURCE_DIR_OVERRIDE is not None:
    SOURCE_DIR = Path(SOURCE_DIR_OVERRIDE).expanduser()
elif SOURCE_FULL_ZIP_OVERRIDE is not None:
    SOURCE_DIR = _extract_full_archive(SOURCE_FULL_ZIP_OVERRIDE)
else:
    candidates = _source_candidates()
    if len(candidates) == 1:
        SOURCE_DIR = candidates[0]
    elif not candidates:
        zip_roots = [RESULTS_ROOT] + ([Path("/kaggle/input")] if RUNTIME == "Kaggle" else [])
        archives = sorted(set(p for root in zip_roots if root.is_dir()
                              for p in root.rglob(SOURCE_EXP_ID + "_FULL.zip")))
        if len(archives) != 1:
            raise FileNotFoundError("Cannot find exactly one 024 256px FULL archive; set SOURCE_DIR_OVERRIDE or SOURCE_FULL_ZIP_OVERRIDE")
        SOURCE_DIR = _extract_full_archive(archives[0])
    else:
        raise RuntimeError(f"Multiple EXP-024 sources: {candidates}. Set SOURCE_DIR_OVERRIDE")
if not SOURCE_DIR.is_dir(): raise FileNotFoundError(SOURCE_DIR)
print("Source EXP-024:", SOURCE_DIR)
print("EXP-025 output root:", RESULTS_ROOT)


In [ ]:
# Verify the immutable 024 source before reading any Test or Holdout labels.
source_config_path = SOURCE_DIR / "experiment_config.json"
source_lock_path = SOURCE_DIR / "validation_selection_lock.json"
source_validation_path = SOURCE_DIR / "aggregate" / "validation_results_5seeds.csv"
for path in (source_config_path, source_lock_path, source_validation_path):
    if not path.is_file(): raise FileNotFoundError(path)
source_config = json.loads(source_config_path.read_text(encoding="utf-8"))
selection_lock = json.loads(source_lock_path.read_text(encoding="utf-8"))
if source_config["experiment_id"] != SOURCE_EXP_ID:
    raise ValueError("Source is not the pinned EXP-024 256px run")
if SOURCE_DIR.name != source_config["experiment_id"]:
    raise ValueError("Source folder and config experiment ID differ")
if source_config["dataset"]["archive"] != DATASET_ARCHIVE_NAME or source_config["dataset"]["archive_sha256"] != EXPECTED_ZIP_SHA256:
    raise ValueError("Source 024 is not the frozen Split (11) archive")
if source_config["dataset"]["counts"] != EXPECTED_TOTALS or source_config["dataset"]["class_names"] != EXPECTED_CLASSES:
    raise ValueError("Source class order or split counts changed")
if source_config["dataset"]["real_filename_counts"] != EXPECTED_REAL_FILENAME_COUNTS:
    raise ValueError("Source real_ counts changed")
if source_config["seeds"] != SEEDS or "256x256" not in source_config["input"]:
    raise ValueError("Source 024 is not the five-seed 256px run")
if source_config["model"]["backbone"] != "DenseNet121" or source_config["model"]["augmentation"].get("brightness_factor") != 0.08:
    raise ValueError("Source model recipe differs from EXP-024")
if selection_lock["experiment_id"] != source_config["experiment_id"] or selection_lock["config_sha256"] != file_sha256(source_config_path):
    raise ValueError("Source Validation lock does not match source config")
if selection_lock["dataset_sha256"] != EXPECTED_ZIP_SHA256 or selection_lock["dataset_manifest_sha256"] != source_config["dataset"]["manifest_sha256"]:
    raise ValueError("Source Validation lock dataset fingerprint changed")
if selection_lock["class_names"] != EXPECTED_CLASSES or selection_lock["seeds"] != SEEDS:
    raise ValueError("Source Validation lock class order/seeds changed")
if selection_lock["validation_sha256"] != file_sha256(source_validation_path):
    raise ValueError("Source Validation CSV changed")
validation_results_df = pd.read_csv(source_validation_path)
if validation_results_df["seed"].tolist() != SEEDS:
    raise ValueError("Source Validation lacks the five expected seeds")
ranked = validation_results_df.sort_values(["val_loss", "val_macro_f1", "seed"], ascending=[True, False, True])
deployment_seed = int(ranked.iloc[0]["seed"])
if deployment_seed != int(selection_lock["deployment_seed"]):
    raise ValueError("Validation-selected deployment seed changed")
for seed in SEEDS:
    checkpoint = SOURCE_DIR / f"seed_{seed}" / "best.weights.h5"
    stage_path = SOURCE_DIR / f"seed_{seed}" / "stage_selection.json"
    if not checkpoint.is_file() or not stage_path.is_file():
        raise FileNotFoundError(f"Missing EXP-024 checkpoint or stage selection for seed {seed}")
    if file_sha256(checkpoint) != selection_lock["selected_checkpoint_sha256"][str(seed)]:
        raise ValueError(f"EXP-024 checkpoint SHA256 changed for seed {seed}")

IMG_SIZE = 256
BATCH_SIZE = int(source_config["batch_size"])
HEAD_DROPOUT = float(source_config["model"]["head_dropout"])
WEIGHT_DECAY = float(source_config["weight_decay"])
EXPECTED_MANIFEST_SHA256 = source_config["dataset"]["manifest_sha256"]
DATASET_ZIP_SHA256 = EXPECTED_ZIP_SHA256
CLASS_NAMES = EXPECTED_CLASSES
NUM_CLASSES = len(CLASS_NAMES)

OUT_DIR = RESULTS_ROOT / EXPERIMENT_ID
AGGREGATE_DIR = OUT_DIR / "aggregate"
PANEL_DIR = OUT_DIR / "figures" / "panels"
for directory in (OUT_DIR, AGGREGATE_DIR, PANEL_DIR): directory.mkdir(parents=True, exist_ok=True)
provenance = {
    "experiment_id": EXPERIMENT_ID,
    "source_experiment_id": source_config["experiment_id"],
    "source_config_sha256": file_sha256(source_config_path),
    "source_validation_lock_sha256": file_sha256(source_lock_path),
    "source_validation_sha256": file_sha256(source_validation_path),
    "source_checkpoint_sha256": selection_lock["selected_checkpoint_sha256"],
    "dataset_zip_sha256": EXPECTED_ZIP_SHA256,
    "dataset_manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "holdout_zip_sha256": REAL_HOLDOUT_ARCHIVE_SHA256,
    "deployment_seed": deployment_seed,
    "training": "NONE", "final_test": RUN_FINAL_TEST, "holdout": RUN_REAL_HOLDOUT,
}
provenance_path = OUT_DIR / "source_024_provenance.json"
if provenance_path.exists() and json.loads(provenance_path.read_text(encoding="utf-8")) != provenance:
    raise ValueError("EXP-025 already points to a different EXP-024 source")
save_json(provenance_path, provenance)
print("Verified 024 Validation lock and five checkpoint hashes; selected seed:", deployment_seed)


## Audit the frozen Split (11) ZIP

In [ ]:
if not VIEW_ONLY:
    # ============================================================
    # 05. LOAD FROZEN SPLIT (8) FROM THE EXACT ZIP
    # ============================================================

    # EXPECTED_ZIP_SHA256 is defined in the configuration cell.

    if DATASET_ARCHIVE_OVERRIDE is not None:
        archive_for_hash = Path(DATASET_ARCHIVE_OVERRIDE)
    elif RUNTIME == "Windows":
        archive_for_hash = Path(r"D:\Study\KhoaLuanTotNghiep\Dataset") / DATASET_ARCHIVE_NAME
    elif RUNTIME == "Colab":
        archive_for_hash = DRIVE_DATASET_ARCHIVE
    else:  # Kaggle input dataset should contain the original Split (11) ZIP.
        archive_candidates = sorted(Path("/kaggle/input").rglob(DATASET_ARCHIVE_NAME))
        if len(archive_candidates) != 1:
            raise RuntimeError(
                f"Expected one {DATASET_ARCHIVE_NAME} under /kaggle/input; "
                f"found: {archive_candidates}. Set DATASET_ARCHIVE_OVERRIDE if needed."
            )
        archive_for_hash = archive_candidates[0]

    if not archive_for_hash.is_file():
        raise FileNotFoundError(
            f"Split (11) ZIP missing: {archive_for_hash}. "
            "Upload/mount the exact archive or set DATASET_ARCHIVE_OVERRIDE."
        )

    digest = hashlib.sha256()
    with archive_for_hash.open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    DATASET_ZIP_SHA256 = digest.hexdigest()
    if DATASET_ZIP_SHA256 != EXPECTED_ZIP_SHA256:
        raise ValueError(
            f"Split (11) ZIP SHA256 mismatch: {DATASET_ZIP_SHA256}; "
            f"expected {EXPECTED_ZIP_SHA256}"
        )
    print("Verified Split (11) ZIP:", archive_for_hash)
    print("Dataset ZIP SHA256:", DATASET_ZIP_SHA256)

    # Keep this cache separate from the Split (7) dataset used by version 011.
    cache_parent = PROJECT_ROOT / "dataset_cache"
    extraction_root = cache_parent / "split_11"
    staging_root = cache_parent / "split_11_staging"
    DATASET_PATH = extraction_root / "CashewData_Split"
    marker = extraction_root / ".archive_sha256"

    if extraction_root.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != EXPECTED_ZIP_SHA256:
            raise RuntimeError(
                f"Split (11) cache has no matching completion marker: {extraction_root}. "
                "Inspect it and remove or rename it before retrying."
            )
    else:
        if staging_root.exists():
            raise RuntimeError(
                f"Previous Split (11) extraction is incomplete: {staging_root}. "
                "Inspect it and remove or rename it before retrying."
            )
        cache_parent.mkdir(parents=True, exist_ok=True)
        staging_root.mkdir()
        with zipfile.ZipFile(archive_for_hash) as dataset_archive:
            for member in dataset_archive.infolist():
                target = (staging_root / member.filename).resolve()
                if target != staging_root.resolve() and staging_root.resolve() not in target.parents:
                    raise ValueError(f"Unsafe path in dataset archive: {member.filename}")
                if not member.filename.startswith("CashewData_Split/"):
                    raise ValueError(f"Unexpected Split (11) ZIP root: {member.filename}")
            print("Extracting Split (11) into:", staging_root)
            dataset_archive.extractall(staging_root)
        if not all((staging_root / "CashewData_Split" / name).is_dir()
                   for name in ("train", "val", "test")):
            raise RuntimeError("Split (11) ZIP has no complete train/val/test directory tree.")
        (staging_root / ".archive_sha256").write_text(EXPECTED_ZIP_SHA256 + "\n", encoding="utf-8")
        staging_root.rename(extraction_root)

    required_splits = [DATASET_PATH / name for name in ("train", "val", "test")]
    missing_splits = [str(path) for path in required_splits if not path.is_dir()]
    if missing_splits:
        raise FileNotFoundError(f"Split (11) cache is incomplete: {missing_splits}")

    TRAIN_DIR = DATASET_PATH / "train"
    VAL_DIR = DATASET_PATH / "val"
    TEST_DIR = DATASET_PATH / "test"

    unexpected_removed_files = [
        DATASET_PATH / Path(relative_path)
        for relative_path in REQUIRED_ABSENT_IMAGES
        if (DATASET_PATH / Path(relative_path)).is_file()
    ]
    if unexpected_removed_files:
        raise ValueError(
            "Split (11) contains files that should be absent:\n" +
            "\n".join(str(path) for path in unexpected_removed_files)
        )

    print("\nSelected dataset:", DATASET_PATH)
    print("TRAIN_DIR:", TRAIN_DIR)
    print("VAL_DIR  :", VAL_DIR)
    print("TEST_DIR :", TEST_DIR)
else:
    print('Cell 6 skipped in VIEW_ONLY')


In [ ]:
if not VIEW_ONLY:

    # ============================================================
    # 06. DATASET AUDIT + EXACT COUNTS
    # ============================================================

    VALID_EXTENSIONS = {
        ".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"
    }


    def sha256_file(path):
        digest = hashlib.sha256()
        with open(path, "rb") as stream:
            for block in iter(lambda: stream.read(1 << 20), b""):
                digest.update(block)
        return digest.hexdigest()


    from PIL import ImageOps
    from scipy.fftpack import dct


    def phash64(image):
        gray = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS), dtype=np.float32)
        frequencies = dct(dct(gray, axis=0, norm="ortho"), axis=1, norm="ortho")[:8, :8]
        bits = frequencies > np.median(frequencies)
        return int("".join("1" if bit else "0" for bit in bits.flat), 2)


    def pixel_hash(image):
        rgb = image.convert("RGB")
        digest = hashlib.sha256()
        digest.update(f"{rgb.width}x{rgb.height}:".encode())
        digest.update(rgb.tobytes())
        return digest.hexdigest()


    def audit_split(split_name, split_dir):
        valid_rows = []
        bad_rows = []
        unknown_rows = []

        found_classes = sorted([
            p.name for p in split_dir.iterdir()
            if p.is_dir()
        ])

        if found_classes != sorted(EXPECTED_CLASSES):
            raise ValueError(
                f"Class folders của {split_name} không đúng.\n"
                f"Expected: {sorted(EXPECTED_CLASSES)}\n"
                f"Found   : {found_classes}"
            )

        for label, class_name in enumerate(EXPECTED_CLASSES):
            class_dir = split_dir / class_name

            for path in sorted(class_dir.rglob("*")):
                if not path.is_file():
                    continue

                ext = path.suffix.lower()

                if ext not in VALID_EXTENSIONS:
                    unknown_rows.append({
                        "split": split_name,
                        "class": class_name,
                        "path": str(path),
                        "extension": ext,
                    })
                    continue

                try:
                    with Image.open(path) as img:
                        img.verify()

                    with Image.open(path) as img:
                        image = ImageOps.exif_transpose(img)
                        image.load()
                        pixels_sha = pixel_hash(image)
                        perceptual_hash = phash64(image)

                    valid_rows.append({
                        "split": split_name,
                        "class": class_name,
                        "label": label,
                        "path": str(path),
                        "relative_path": path.relative_to(DATASET_PATH).as_posix(),
                        "sha256": sha256_file(path),
                        "pixel_sha256": pixels_sha,
                        "phash64": perceptual_hash,
                        "is_real_filename": path.name.casefold().startswith("real_"),
                    })

                except Exception as e:
                    bad_rows.append({
                        "split": split_name,
                        "class": class_name,
                        "path": str(path),
                        "error": str(e),
                    })

        return valid_rows, bad_rows, unknown_rows


    all_valid = []
    all_bad = []
    all_unknown = []

    for split_name, split_dir in {
        "train": TRAIN_DIR,
        "val": VAL_DIR,
        "test": TEST_DIR,
    }.items():

        print(f"Checking {split_name}...")

        valid_rows, bad_rows, unknown_rows = audit_split(
            split_name,
            split_dir
        )

        all_valid.extend(valid_rows)
        all_bad.extend(bad_rows)
        all_unknown.extend(unknown_rows)


    valid_df = pd.DataFrame(all_valid)
    bad_df = pd.DataFrame(all_bad)
    unknown_df = pd.DataFrame(all_unknown)

    print("\n" + "=" * 70)
    print("VALID IMAGES   :", len(valid_df))
    print("INVALID IMAGES :", len(bad_df))
    print("UNKNOWN FILES  :", len(unknown_df))
    print("=" * 70)

    if len(bad_df):
        display(bad_df)

    if len(unknown_df):
        display(unknown_df)

    bad_df.to_csv(OUT_DIR / "invalid_images.csv", index=False)
    unknown_df.to_csv(OUT_DIR / "unknown_files.csv", index=False)

    if len(bad_df):
        raise ValueError("Dataset còn ảnh lỗi. Hãy clean dataset trước khi train.")

    if len(unknown_df):
        raise ValueError("Dataset có file extension lạ. Hãy kiểm tra trước khi train.")


    stats_df = (
        valid_df.groupby(["class", "split"])
        .size()
        .unstack(fill_value=0)
        .reindex(EXPECTED_CLASSES)
    )

    for col in ["train", "val", "test"]:
        if col not in stats_df.columns:
            stats_df[col] = 0

    stats_df = stats_df[["train", "val", "test"]]
    stats_df["total"] = stats_df.sum(axis=1)

    display(stats_df)

    actual_totals = {
        "train": int(stats_df["train"].sum()),
        "val": int(stats_df["val"].sum()),
        "test": int(stats_df["test"].sum()),
        "total": int(stats_df["total"].sum()),
    }

    mismatches = []

    for class_name in EXPECTED_CLASSES:
        for split_name in ["train", "val", "test", "total"]:

            actual = int(stats_df.loc[class_name, split_name])
            expected = int(EXPECTED_COUNTS[class_name][split_name])

            if actual != expected:
                mismatches.append({
                    "class": class_name,
                    "split": split_name,
                    "actual": actual,
                    "expected": expected,
                })

    for split_name in ["train", "val", "test", "total"]:
        if actual_totals[split_name] != EXPECTED_TOTALS[split_name]:
            mismatches.append({
                "class": "TOTAL",
                "split": split_name,
                "actual": actual_totals[split_name],
                "expected": EXPECTED_TOTALS[split_name],
            })

    if mismatches:
        mismatch_df = pd.DataFrame(mismatches)
        display(mismatch_df)

        if STRICT_DATASET_CHECK:
            raise ValueError(
                "Dataset count does not match the frozen Split (11) archive."
            )
    else:
        print("Dataset counts match the frozen Split (11) archive.")
        print(actual_totals)


    # Save manifest để biết chính xác file nào thuộc split nào.
    valid_df.to_csv(
        OUT_DIR / "dataset_manifest.csv",
        index=False
    )


    manifest_records = valid_df[["relative_path", "label", "sha256"]].to_dict("records")
    DATASET_MANIFEST_SHA256 = hashlib.sha256(
        json.dumps(manifest_records, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    (OUT_DIR / "dataset_manifest_sha256.txt").write_text(DATASET_MANIFEST_SHA256 + "\n", encoding="utf-8")
    print("Dataset manifest SHA256:", DATASET_MANIFEST_SHA256)
    if EXPECTED_MANIFEST_SHA256 is not None and DATASET_MANIFEST_SHA256 != EXPECTED_MANIFEST_SHA256:
        raise ValueError("Frozen Split (11) manifest SHA256 mismatch")


    real_counts = (valid_df.groupby("split")["is_real_filename"]
                   .agg(["sum", "count"]).reindex(["train", "val", "test"]))
    actual_real_counts = {s: int(real_counts.loc[s, "sum"]) for s in ("train", "val", "test")}
    if actual_real_counts != EXPECTED_REAL_FILENAME_COUNTS:
        raise ValueError(
            f"Split (11) real_ filename counts mismatch: {actual_real_counts}; "
            f"expected {EXPECTED_REAL_FILENAME_COUNTS}"
        )
    real_counts.to_csv(OUT_DIR / "real_filename_counts_by_split.csv")
    real_class_counts = (valid_df.groupby(["split", "class"])["is_real_filename"]
                         .agg(["sum", "count"]))
    real_class_counts.to_csv(OUT_DIR / "real_filename_counts_by_split_class.csv")
    print("real_ filename counts:")
    display(real_counts)
    if int(real_counts.loc["val", "sum"]) == 0:
        print("WARNING: Validation has no real_ filename; checkpoints cannot be selected for this group.")

    exact_rows = []
    for digest_column in ("sha256", "pixel_sha256"):
        for _, group in valid_df.groupby(digest_column, sort=False):
            if len(group) < 2:
                continue
            records = group[["relative_path", "split", "class"]].to_dict("records")
            for a_index in range(len(records)):
                for b_index in range(a_index + 1, len(records)):
                    a, b = records[a_index], records[b_index]
                    if a["split"] != b["split"] or a["class"] != b["class"]:
                        exact_rows.append({"kind": digest_column, "a": a["relative_path"],
                                           "b": b["relative_path"], "a_split": a["split"],
                                           "b_split": b["split"], "a_class": a["class"],
                                           "b_class": b["class"]})
    exact_cross_df = pd.DataFrame(exact_rows).drop_duplicates() if exact_rows else pd.DataFrame(
        columns=["kind", "a", "b", "a_split", "b_split", "a_class", "b_class"])
    exact_cross_df.to_csv(OUT_DIR / "exact_cross_split_or_label_pairs.csv", index=False)
    if len(exact_cross_df):
        raise RuntimeError("Byte/pixel-identical cross-split or cross-label files found; inspect exact_cross_split_or_label_pairs.csv")


    def hamming64(a, b):
        return (int(a) ^ int(b)).bit_count()


    def thumbnail_gray(path):
        with Image.open(path) as img:
            image = ImageOps.exif_transpose(img).convert("L")
            return np.asarray(image.resize((64, 64), Image.Resampling.BILINEAR), dtype=np.float32)


    class BKNode:
        def __init__(self, value, row):
            self.value, self.rows, self.children = int(value), [row], {}

        def insert(self, value, row):
            distance = hamming64(self.value, value)
            if distance == 0:
                self.rows.append(row)
            elif distance in self.children:
                self.children[distance].insert(value, row)
            else:
                self.children[distance] = BKNode(value, row)

        def query(self, value, radius, results):
            distance = hamming64(self.value, value)
            if distance <= radius:
                results.extend((row, distance) for row in self.rows)
            for edge, child in self.children.items():
                if distance - radius <= edge <= distance + radius:
                    child.query(value, radius, results)


    PHASH_MAX_DISTANCE = 4
    tree = None
    near_rows = []
    thumbnail_cache = {}
    audit_records = valid_df.to_dict("records")
    for row_index, row in enumerate(audit_records):
        value = int(row["phash64"])
        if tree is None:
            tree = BKNode(value, row_index)
            continue
        matches = []
        tree.query(value, PHASH_MAX_DISTANCE, matches)
        for other_index, distance in matches:
            other = audit_records[other_index]
            if row["split"] == other["split"] and row["class"] == other["class"]:
                continue
            for index in (row_index, other_index):
                if index not in thumbnail_cache:
                    thumbnail_cache[index] = thumbnail_gray(audit_records[index]["path"])
            mad = float(np.abs(thumbnail_cache[row_index] - thumbnail_cache[other_index]).mean())
            near_rows.append({"a": row["relative_path"], "b": other["relative_path"],
                              "a_split": row["split"], "b_split": other["split"],
                              "a_class": row["class"], "b_class": other["class"],
                              "phash_hamming": distance, "mad_64_gray": mad,
                              "thumbnail_match_candidate": mad <= 1.0})
        tree.insert(value, row_index)

    near_df = pd.DataFrame(near_rows, columns=["a", "b", "a_split", "b_split", "a_class",
                                                "b_class", "phash_hamming", "mad_64_gray",
                                                "thumbnail_match_candidate"])
    near_df.to_csv(OUT_DIR / "near_cross_split_or_label_pairs.csv", index=False)
    audit_summary = {
        "dataset_zip_sha256": DATASET_ZIP_SHA256,
        "total_images": int(len(valid_df)),
        "split_counts": actual_totals,
        "real_filename_counts": {s: int(real_counts.loc[s, "sum"]) for s in ("train", "val", "test")},
        "exact_cross_split_or_label_pairs": int(len(exact_cross_df)),
        "phash_cross_split_or_label_candidates": int(len(near_df)),
        "thumbnail_match_candidates": int(near_df["thumbnail_match_candidate"].sum()),
        "note": "pHash/thumbnail pairs require manual review; real_ is a filename proxy, not verified capture source.",
    }
    (OUT_DIR / "dataset_audit_summary.json").write_text(
        json.dumps(audit_summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Near-duplicate review candidates:", len(near_df))
    display(near_df[near_df["thumbnail_match_candidate"]].head(20))
    if audit_summary["thumbnail_match_candidates"]:
        print("WARNING: visually matching cross-split/cross-label candidates exist; review before treating Test as independent.")
else:
    print('Cell 8 skipped in VIEW_ONLY')


In [ ]:
if DATASET_MANIFEST_SHA256 != EXPECTED_MANIFEST_SHA256:
    raise ValueError("Split (11) file manifest differs from trained EXP-024 dataset")
print("Verified source dataset manifest:", DATASET_MANIFEST_SHA256)


In [ ]:
if not VIEW_ONLY:

    # ============================================================
    # 04. REPRODUCIBILITY
    # ============================================================

    def set_global_seed(seed: int):
        os.environ["PYTHONHASHSEED"] = str(seed)

        random.seed(seed)
        np.random.seed(seed)
        tf.random.set_seed(seed)

        # Đồng bộ Python / NumPy / TensorFlow theo Keras.
        try:
            keras.utils.set_random_seed(seed)
        except Exception:
            pass

        print(f"Global seed set to: {seed}")


    try:
        tf.config.experimental.enable_op_determinism()
        print("Deterministic operations: ENABLED")
    except Exception as e:
        print("Deterministic operations: unavailable")
        print("Reason:", e)
else:
    print('Cell 5 skipped in VIEW_ONLY')


In [ ]:
if not VIEW_ONLY:

    # ============================================================
    # 08. FIXED FILE LISTS + TF.DATA
    # ============================================================

    CLASS_NAMES = EXPECTED_CLASSES
    NUM_CLASSES = len(CLASS_NAMES)
    AUTOTUNE = tf.data.AUTOTUNE


    def split_arrays(split_name):
        df = (
            valid_df[valid_df["split"] == split_name]
            .copy()
            .reset_index(drop=True)
        )

        return (
            df["path"].to_numpy(),
            df["label"].to_numpy(dtype=np.int32),
            df
        )


    train_paths, train_labels, train_meta = split_arrays("train")
    val_paths, val_labels, val_meta = split_arrays("val")
    test_paths, test_labels, test_meta = split_arrays("test")


    def decode_image(path, label):
        image_bytes = tf.io.read_file(path)

        image = tf.io.decode_image(
            image_bytes,
            channels=3,
            expand_animations=False
        )

        image.set_shape([None, None, 3])

        image = tf.image.resize(
            image,
            [IMG_SIZE, IMG_SIZE],
            method="bilinear"
        )

        return tf.cast(image, tf.float32), label


    def make_dataset(paths, labels, seed, training=False):
        ds = tf.data.Dataset.from_tensor_slices(
            (paths, labels)
        )

        if training:
            ds = ds.shuffle(
                buffer_size=len(paths),
                seed=seed,
                reshuffle_each_iteration=True
            )

        ds = ds.map(
            decode_image,
            num_parallel_calls=AUTOTUNE
        )

        ds = ds.batch(
            BATCH_SIZE,
            drop_remainder=False
        )

        ds = ds.prefetch(AUTOTUNE)

        return ds


    print("Fixed split:")
    print("Train :", len(train_paths))
    print("Val   :", len(val_paths))
    print("Test  :", len(test_paths))

    print("\nClass mapping:")
    for idx, class_name in enumerate(CLASS_NAMES):
        print(f"{idx}: {class_name}")
else:
    print('Cell 12 skipped in VIEW_ONLY')


## Rebuild the architecture and load only frozen 024 weights

In [ ]:
if not VIEW_ONLY:
    # 08. ImageNet backbone, frozen BN, and smoke test
    import tempfile
    from PIL import ImageOps

    BRIGHTNESS_FACTOR = 0.08
    BRIGHTNESS_VALUE_RANGE = (0.0, 255.0)
    if not hasattr(layers, "RandomBrightness"):
        raise RuntimeError("This Keras runtime lacks RandomBrightness")
    AUGMENTATION_CONFIG = {"horizontal_flip": True, "rotation_factor": 0.08,
                           "zoom_factor": 0.10, "contrast_factor": 0.10,
                           "brightness_enabled": True, "brightness_factor": BRIGHTNESS_FACTOR,
                           "brightness_value_range": list(BRIGHTNESS_VALUE_RANGE)}

    def densenet_preprocessing(inputs):
        x = layers.Rescaling(1.0 / 255.0, name="densenet_rescale")(inputs)
        return layers.Normalization(mean=[0.485, 0.456, 0.406],
                                    variance=[0.229**2, 0.224**2, 0.225**2],
                                    axis=-1, name="densenet_preprocess")(x)

    def build_transfer_model(seed, weights="imagenet"):
        set_global_seed(seed)
        inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="raw_rgb")
        augmented = keras.Sequential([
            layers.RandomFlip("horizontal", seed=seed),
            layers.RandomRotation(0.08, seed=seed),
            layers.RandomZoom(0.10, seed=seed),
            layers.RandomContrast(0.10, seed=seed),
            layers.RandomBrightness(factor=BRIGHTNESS_FACTOR,
                                    value_range=BRIGHTNESS_VALUE_RANGE, seed=seed + 1100),
        ], name=f"train_only_augmentation_seed_{seed}")(inputs)
        x = densenet_preprocessing(augmented)
        backbone = keras.applications.DenseNet121(include_top=False, weights=weights,
                                                   input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
        backbone.trainable = False
        features = backbone(x, training=False)  # Keep BN moving statistics fixed in both stages.
        x = layers.Dropout(HEAD_DROPOUT, seed=seed)(features)
        outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32",
                               name="disease_output")(x)
        return keras.Model(inputs, outputs, name=f"DenseNet121_Transfer_seed_{seed}"), backbone

    def set_backbone_stage(backbone, stage):
        if stage == "stage1":
            backbone.trainable = False
        elif stage == "stage2":
            backbone.trainable = True
            for layer in backbone.layers:
                layer.trainable = (layer.name.startswith("conv5_") and
                                   not isinstance(layer, layers.BatchNormalization))
            if not any(layer.trainable and layer.weights for layer in backbone.layers):
                raise RuntimeError("No weighted conv5_* layer was unfrozen")
            if any(layer.trainable for layer in backbone.layers
                   if isinstance(layer, layers.BatchNormalization)):
                raise RuntimeError("A backbone BatchNorm layer is trainable")
        else:
            raise ValueError(stage)
        return int(sum(np.prod(v.shape) for v in backbone.trainable_weights))

    def compile_stage(model, lr):
        model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr,
                                                      weight_decay=WEIGHT_DECAY),
                      loss=keras.losses.SparseCategoricalCrossentropy(),
                      metrics=["accuracy"], steps_per_execution=1, jit_compile=False)

    def save_json(path, value):
        path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")

    def file_sha256(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as stream:
            for block in iter(lambda: stream.read(1 << 20), b""):
                digest.update(block)
        return digest.hexdigest()



## Final Test — five frozen EXP-024 models

In [12]:
# 18. Guarded Final Test
def assert_selection_lock():
    saved = json.loads((SOURCE_DIR / "validation_selection_lock.json").read_text(encoding="utf-8"))
    if saved != selection_lock or saved["config_sha256"] != file_sha256(source_config_path):
        raise ValueError("Frozen Validation/config lock changed")
    for seed in SEEDS:
        actual = file_sha256(SOURCE_DIR / f"seed_{seed}" / "best.weights.h5")
        if actual != saved["selected_checkpoint_sha256"][str(seed)]:
            raise ValueError(f"Seed {seed} checkpoint changed after Validation lock")
    return saved

def load_selected_model(seed):
    # weights=None is only for reconstructing architecture before immediately
    # restoring the EXP-024 trained checkpoint. Training always uses ImageNet.
    model, backbone = build_transfer_model(seed, weights=None)
    selected = json.loads((SOURCE_DIR / f"seed_{seed}" / "stage_selection.json").read_text(
        encoding="utf-8"))
    set_backbone_stage(backbone, selected["selected_stage"])
    model.load_weights(SOURCE_DIR / f"seed_{seed}" / "best.weights.h5")
    return model

if RUN_FINAL_TEST and (AGGREGATE_DIR / "test_results_5seeds.csv").is_file():
    assert_selection_lock()
    test_results_df = pd.read_csv(AGGREGATE_DIR / "test_results_5seeds.csv")
    if test_results_df["seed"].tolist() != SEEDS or any(
            not (OUT_DIR / f"seed_{seed}" / "test" / "predictions.csv").is_file()
            for seed in SEEDS):
        raise RuntimeError("Existing Final Test artifacts are incomplete.")
    print("Final Test already completed; reading frozen results.")
elif RUN_FINAL_TEST:
    assert_selection_lock()
    if any((OUT_DIR / f"seed_{seed}" / "test" / "metrics.json").exists() for seed in SEEDS):
        raise RuntimeError("Partial Test exists; inspect before retrying to avoid re-scoring.")
    test_ds = make_dataset(test_paths, test_labels, SEEDS[0], training=False)
    test_rows = []
    for seed in SEEDS:
        keras.backend.clear_session(); gc.collect()
        model = load_selected_model(seed)
        output = OUT_DIR / f"seed_{seed}" / "test"
        output.mkdir(parents=True, exist_ok=True)
        _ = model.predict(test_ds.take(1), verbose=0)  # Warm up tracing.
        started = time.perf_counter()
        probability = np.asarray(model.predict(test_ds, verbose=0))
        elapsed = time.perf_counter() - started
        y_true = np.asarray(test_labels, dtype=np.int32)
        y_pred = probability.argmax(axis=1)
        if probability.shape != (len(y_true), NUM_CLASSES):
            raise RuntimeError("Final Test prediction count mismatch")
        test_row = {
            "seed": seed, "images": len(y_true),
            "loss": float(-np.log(np.clip(probability[np.arange(len(y_true)), y_true], 1e-7, 1)).mean()),
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
            "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "prediction_seconds_including_input_pipeline": elapsed,
            "inference_time_ms_per_image": 1000 * elapsed / len(y_true),
        }
        for group_name, mask in (("real_filename", test_meta["is_real_filename"].to_numpy(dtype=bool)),
                                 ("other_filename", ~test_meta["is_real_filename"].to_numpy(dtype=bool))):
            test_row[f"{group_name}_correct"] = int((y_true[mask] == y_pred[mask]).sum())
            test_row[f"{group_name}_total"] = int(mask.sum())
            test_row[f"{group_name}_accuracy"] = float(accuracy_score(y_true[mask], y_pred[mask]))
        save_json(output / "metrics.json", test_row)
        pd.DataFrame(classification_report(y_true, y_pred, labels=np.arange(NUM_CLASSES),
                                           target_names=CLASS_NAMES, output_dict=True,
                                           zero_division=0)).T.to_csv(output / "classification_report.csv")
        predictions = pd.DataFrame({"filename": test_meta["path"],
                                    "is_real_filename": test_meta["is_real_filename"],
                                    "true_label": [CLASS_NAMES[i] for i in y_true],
                                    "predicted_label": [CLASS_NAMES[i] for i in y_pred],
                                    "confidence": probability.max(axis=1),
                                    "correct": y_true == y_pred})
        for index, name in enumerate(CLASS_NAMES):
            predictions[f"prob_{name}"] = probability[:, index]
        predictions.to_csv(output / "predictions.csv", index=False)
        predictions[~predictions["correct"]].to_csv(output / "error_cases.csv", index=False)
        cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
        for normalized, suffix in ((False, ""), (True, "_normalized")):
            matrix = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES),
                                      normalize="true" if normalized else None)
            pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
                output / f"confusion_matrix{suffix}.csv")
        test_rows.append(test_row)
        del model
    test_results_df = pd.DataFrame(test_rows)
    test_results_df.to_csv(AGGREGATE_DIR / "test_results_5seeds.csv", index=False)
    pd.DataFrame([{"metric": col, "mean": test_results_df[col].mean(),
                   "std": test_results_df[col].std(ddof=1)}
                  for col in ("accuracy", "macro_precision", "macro_recall", "macro_f1",
                              "balanced_accuracy", "inference_time_ms_per_image")]).to_csv(
        AGGREGATE_DIR / "test_mean_std_summary.csv", index=False)
    for normalized, suffix in ((False, ""), (True, "_normalized")):
        fig, axes = plt.subplots(1, len(SEEDS), figsize=(28, 5.5), facecolor="white")
        for column, seed in enumerate(SEEDS):
            matrix = pd.read_csv(OUT_DIR / f"seed_{seed}" / "test" /
                                 f"confusion_matrix{suffix}.csv", index_col=0).to_numpy()
            ax = axes[column]
            ax.imshow(matrix, cmap=MATRIX_CMAP, vmin=0,
                      vmax=1 if normalized else max(1, matrix.max()))
            ax.set_title(f"Seed {seed}")
            ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
            ax.set_yticks(range(NUM_CLASSES))
            ax.set_yticklabels(CLASS_NAMES if column == 0 else [])
            for row in range(NUM_CLASSES):
                for col in range(NUM_CLASSES):
                    ax.text(col, row, f"{matrix[row, col]:.2f}" if normalized else str(int(matrix[row, col])),
                            ha="center", va="center", color="black", fontsize=8)
            ax.set_xlabel("Predicted")
        axes[0].set_ylabel("True")
        fig.tight_layout()
        fig.savefig(PANEL_DIR / f"test_confusion_matrices{suffix}_5seeds_panel.png",
                    dpi=190, bbox_inches="tight")
        plt.close(fig)
    display(test_results_df)
else:
    print("Final Test PENDING (RUN_FINAL_TEST=False)")


Global seed set to: 42
Global seed set to: 123
Global seed set to: 2026
Global seed set to: 3407
Global seed set to: 7777


,seed,images,loss,accuracy,macro_precision,macro_recall,macro_f1,balanced_accuracy,prediction_seconds_including_input_pipeline,inference_time_ms_per_image,real_filename_correct,real_filename_total,real_filename_accuracy,other_filename_correct,other_filename_total,other_filename_accuracy
0,42,685,0.122153,0.957664,0.953562,0.953585,0.953527,0.953585,1.672408,2.441472,26,28,0.928571,630,657,0.958904
1,123,685,0.122853,0.957664,0.954350,0.952975,0.953495,0.952975,1.607844,2.347217,25,28,0.892857,631,657,0.960426
2,2026,685,0.118079,0.960584,0.957527,0.956605,0.956932,0.956605,1.611478,2.352523,25,28,0.892857,633,657,0.963470
3,3407,685,0.111794,0.967883,0.965158,0.964570,0.964818,0.964570,1.615277,2.358068,26,28,0.928571,637,657,0.969559
4,7777,685,0.113768,0.960584,0.957155,0.956337,0.956693,0.956337,1.611637,2.352755,26,28,0.928571,632,657,0.961948


## Holdout (5) — separate four-class evaluation

In [13]:
"""External classification evaluation for EXP-024.

This file has no training code. Holdout labels never select checkpoints or seeds.
"""

import hashlib
import json
import os
import stat
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score


REAL_HOLDOUT_HELPER_VERSION = "006.1"
REAL_HOLDOUT_CMAP = LinearSegmentedColormap.from_list(
    "real_holdout_readable_blues", ["#ffffff", "#dbeafe", "#93c5fd"]
)


def holdout_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def resolve_holdout_archive(archive_name, override=None):
    if override is not None:
        candidate = Path(override)
        if not candidate.is_file():
            raise FileNotFoundError(candidate)
        return candidate
    if RUNTIME == "Colab":
        candidate = DRIVE_PROJECT_DIR / "Dataset" / archive_name
        if candidate.is_file():
            return candidate
    if os.name == "nt":
        candidate = Path(r"D:\Study\KhoaLuanTotNghiep\Dataset") / archive_name
        if candidate.is_file():
            return candidate
    if Path("/kaggle/input").is_dir():
        candidates = sorted(Path("/kaggle/input").rglob(archive_name))
        if len(candidates) == 1:
            return candidates[0]
    raise FileNotFoundError(
        f"Cannot uniquely locate {archive_name}. Set REAL_HOLDOUT_ARCHIVE_OVERRIDE "
        "to the mounted ZIP path; on Colab check the project shortcut and original-folder permissions."
    )


def prepare_real_holdout(archive_path, extraction_dir, class_names, expected_sha256,
                         expected_counts, reference_rows, audit_dir):
    """Verify the frozen archive, labels, decoding and byte overlap with Split (11).

    reference_rows: dictionaries with path/split and optional existing sha256.
    Exact-byte checking does not establish independence of near-duplicate photos.
    """
    archive_path, extraction_dir, audit_dir = map(Path, (archive_path, extraction_dir, audit_dir))
    actual_hash = holdout_sha256(archive_path)
    if actual_hash != expected_sha256.upper():
        raise ValueError(f"Holdout archive changed: {actual_hash}; expected {expected_sha256}.")
    marker = extraction_dir / ".archive_sha256"
    if extraction_dir.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != actual_hash:
            raise FileExistsError(f"Unverified extraction: {extraction_dir}; choose a new directory.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            resolved_root = extraction_dir.resolve()
            for member in archive.infolist():
                target = (extraction_dir / member.filename.replace("\\", "/")).resolve()
                is_symlink = stat.S_ISLNK(member.external_attr >> 16)
                if is_symlink or (target != resolved_root and resolved_root not in target.parents):
                    raise ValueError(f"Unsafe ZIP member: {member.filename}")
            extraction_dir.mkdir(parents=True)
            archive.extractall(extraction_dir)
        marker.write_text(actual_hash, encoding="utf-8")

    dataset_root = extraction_dir / "RL_Cashew_holdout"
    if not dataset_root.is_dir():
        raise FileNotFoundError(dataset_root)
    found_classes = sorted(p.name for p in dataset_root.iterdir() if p.is_dir())
    if found_classes != sorted(expected_counts) or not set(found_classes).issubset(class_names):
        raise ValueError(f"Unexpected holdout class folders: {found_classes}")
    rows = []
    for label, name in enumerate(class_names):
        class_dir = dataset_root / name
        files = sorted(p for p in class_dir.rglob("*") if p.is_file()) if class_dir.is_dir() else []
        if len(files) != expected_counts.get(name, 0):
            raise ValueError(f"Unexpected holdout count for {name}: {len(files)}")
        for image_path in files:
            if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                raise ValueError(f"Unsupported holdout image: {image_path}")
            with Image.open(image_path) as image:
                image.verify()
            with Image.open(image_path) as image:
                image.convert("RGB").load()
            rows.append({"path": str(image_path), "relative_path": image_path.relative_to(dataset_root).as_posix(),
                         "class": name, "label": label, "sha256": holdout_sha256(image_path)})
    manifest = pd.DataFrame(rows)
    if manifest.empty or manifest["sha256"].duplicated().any():
        raise ValueError("Holdout is empty or contains duplicate image bytes.")
    # A marker alone is insufficient: verify cached extraction bytes against the ZIP.
    with zipfile.ZipFile(archive_path) as frozen_archive:
        for row in rows:
            digest = hashlib.sha256()
            with frozen_archive.open("RL_Cashew_holdout/" + row["relative_path"]) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest().upper() != row["sha256"]:
                raise ValueError(f"Cached holdout extraction changed: {row['relative_path']}")
    audit_dir.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(audit_dir / "archive_dataset_manifest.csv", index=False)

    references = list(reference_rows)
    if not references or not {"train", "val", "test"}.issubset({r["split"] for r in references}):
        raise ValueError("Overlap audit needs the full train/val/test Split (11) reference manifest.")
    holdout_hashes = set(manifest["sha256"])
    holdout_path_by_hash = dict(zip(manifest["sha256"], manifest["relative_path"]))
    overlaps = []
    reference_digest = hashlib.sha256()
    for row in references:
        digest = row.get("sha256") or holdout_sha256(row["path"])
        digest = digest.upper()
        identity = row.get("relative_path", str(row["path"]))
        reference_digest.update(f"{row['split']}\0{identity}\0{digest}\n".encode("utf-8"))
        if digest in holdout_hashes:
            overlaps.append({"reference_path": str(row["path"]), "split": row["split"],
                             "holdout_relative_path": holdout_path_by_hash[digest],
                             "sha256": digest})
    # Exact overlap is unexpected for this frozen V11/holdout05 pair.
    expected_overlap = set()  # Verified against both local ZIPs before authoring EXP-024.
    observed_overlap = {(row["split"], row["holdout_relative_path"]) for row in overlaps}
    if observed_overlap != expected_overlap:
        raise ValueError(f"Unexpected holdout/Split (11) byte overlaps: {overlaps}")
    evaluated_manifest = manifest[~manifest["sha256"].isin(
        {row["sha256"] for row in overlaps}
    )].reset_index(drop=True)
    evaluated_manifest.to_csv(audit_dir / "dataset_manifest.csv", index=False)
    pd.DataFrame(overlaps).to_csv(audit_dir / "excluded_overlap_images.csv", index=False)
    present = [name for name in class_names if expected_counts.get(name, 0) > 0]
    audit = {"archive_sha256": actual_hash, "archive_name": archive_path.name,
             "archive_manifest_sha256": hashlib.sha256(manifest[["relative_path", "class", "label", "sha256"]]
                  .to_csv(index=False).encode()).hexdigest(),
              "manifest_sha256": hashlib.sha256(evaluated_manifest[["relative_path", "class", "label", "sha256"]]
                 .to_csv(index=False).encode()).hexdigest(),
             "counts": {name: expected_counts.get(name, 0) for name in class_names},
             "images": len(evaluated_manifest), "archive_images": len(manifest),
              "excluded_overlap_images": len(overlaps),
              "evaluated_counts": evaluated_manifest["class"].value_counts().to_dict(),
              "present_classes": present,
             "unsupported_classes": [name for name in class_names if name not in present],
             "reference_images_checked": len(references), "reference_fingerprint": reference_digest.hexdigest(),
             "byte_overlap_count": len(overlaps), "byte_overlaps": overlaps,
             "limitation": "Byte audit only; near-duplicate photos and shared trees/sessions need a separate audit."}
    (audit_dir / "dataset_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print("Real holdout:", len(evaluated_manifest), "evaluated of", len(manifest),
          "archive images; excluded exact Test duplicate:", overlaps)
    print("Present classes:", present)
    print("Not evaluated (no true examples):", audit["unsupported_classes"])
    return evaluated_manifest, audit


def save_real_holdout_seed(manifest, probabilities, class_names, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(manifest), len(class_names)) or not np.isfinite(probabilities).all():
        raise ValueError("Holdout predictions have invalid shape or non-finite values.")
    if (probabilities < 0).any() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):
        raise ValueError("Holdout evaluator expects probabilities, not logits.")
    y_true = manifest["label"].to_numpy(dtype=np.int32)
    y_pred = probabilities.argmax(axis=1)
    present_ids = sorted(set(y_true.tolist()))
    all_ids = list(range(len(class_names)))
    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision_present_classes": float(precision_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "macro_recall_present_classes": float(recall_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "macro_f1_present_classes": float(f1_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "balanced_accuracy_present_classes": float(recall_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "images": len(manifest), "macro_class_count": len(present_ids),
    }
    report = pd.DataFrame(classification_report(y_true, y_pred, labels=all_ids,
        target_names=class_names, output_dict=True, zero_division=0)).transpose()
    per_class = report.loc[class_names, ["precision", "recall", "f1-score", "support"]].copy()
    per_class["evaluated"] = per_class["support"] > 0
    # No evidence for unsupported true classes: don't present placeholder zeros as performance.
    per_class.loc[~per_class["evaluated"], ["precision", "recall", "f1-score"]] = np.nan
    per_class.to_csv(output_dir / "per_class_metrics.csv", index_label="class")
    readable_report = per_class.copy()
    readable_report["scope"] = np.where(readable_report["evaluated"], "true class present", "not evaluated: zero true support")
    readable_report.loc["macro avg (present classes)"] = [
        result["macro_precision_present_classes"], result["macro_recall_present_classes"],
        result["macro_f1_present_classes"], len(manifest), True,
        f"{len(present_ids)} supported true classes only",
    ]
    readable_report.loc["weighted avg"] = [
        report.loc["weighted avg", "precision"], report.loc["weighted avg", "recall"],
        report.loc["weighted avg", "f1-score"], len(manifest), True, "weighted by true support",
    ]
    readable_report.to_csv(output_dir / "classification_report.csv", index_label="class")
    predictions = manifest[["relative_path", "class", "label"]].copy()
    predictions["predicted_label"] = y_pred
    predictions["predicted_class"] = [class_names[i] for i in y_pred]
    predictions["confidence"] = probabilities.max(axis=1)
    predictions["correct"] = y_true == y_pred
    for label, name in enumerate(class_names):
        predictions[f"probability_{name}"] = probabilities[:, label]
    predictions.to_csv(output_dir / "predictions.csv", index=False)
    matrix = confusion_matrix(y_true, y_pred, labels=all_ids)
    for normalized, suffix in ((False, ""), (True, "_normalized")):
        values = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1) if normalized else matrix
        pd.DataFrame(values, index=class_names, columns=class_names).to_csv(output_dir / f"confusion_matrix{suffix}.csv")
        fig, ax = plt.subplots(figsize=(9, 8), facecolor="white")
        ax.set_facecolor("white")
        ax.imshow(values, cmap=REAL_HOLDOUT_CMAP, vmin=0,
                  vmax=1 if normalized else max(int(matrix.max()), 1))
        ax.set_xticks(all_ids, class_names, rotation=40, ha="right", color="black")
        ax.set_yticks(all_ids, class_names, color="black")
        ax.set_xlabel("Predicted class", color="black")
        ax.set_ylabel("True class", color="black")
        ax.set_title("Real holdout" + (" (row normalized)" if normalized else ""), color="black")
        for row in all_ids:
            for col in all_ids:
                text = f"{values[row, col]:.2f}" if normalized else str(int(values[row, col]))
                if normalized and row not in present_ids:
                    text = "N/A"
                ax.text(col, row, text, ha="center", va="center", color="black")
        fig.tight_layout()
        fig.savefig(output_dir / f"confusion_matrix{suffix}.png", dpi=180)
        plt.close(fig)
    (output_dir / "metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


def evaluate_real_holdout(manifest, audit, class_names, seeds, validation_results,
                          checkpoints, load_model, make_dataset, clear_session,
                          output_dir, outputs_are_logits=False):
    """Freeze all checkpoints and validation selection BEFORE any prediction.

    Completed evaluations are read back, not repeated. Interrupted evaluations
    may resume only with the same checkpoint, validation and archive fingerprints.
    """
    output_dir = Path(output_dir)
    seeds = [int(seed) for seed in seeds]
    validation_results = validation_results.copy()
    if validation_results["seed"].tolist() != seeds:
        raise ValueError("Holdout evaluation requires completed, ordered validation for every seed.")
    if not np.isfinite(validation_results[["val_macro_f1", "val_loss"]].to_numpy()).all():
        raise ValueError("Validation selection metrics are incomplete.")
    ranked = validation_results.sort_values(["val_loss", "val_macro_f1", "seed"], ascending=[True, False, True])
    deployment_seed = int(ranked.iloc[0]["seed"])
    checkpoint_hashes = {str(seed): holdout_sha256(checkpoints[seed]) for seed in seeds}
    lock = {"helper_version": REAL_HOLDOUT_HELPER_VERSION, "archive_sha256": audit["archive_sha256"],
            "manifest_sha256": audit["manifest_sha256"],
            "reference_fingerprint": audit["reference_fingerprint"], "seeds": seeds,
            "checkpoint_sha256": checkpoint_hashes, "deployment_seed": deployment_seed,
            # Canonical float precision avoids false lock changes on CSV reload.
            "validation_sha256": hashlib.sha256(validation_results.to_csv(index=False, float_format="%.12g").encode()).hexdigest(),
            "model_classes": class_names, "macro_classes": audit["present_classes"],
            "selection_source": "validation only; lowest val_loss, then macro F1 and seed",
            "outputs_are_logits": outputs_are_logits}
    lock_path = output_dir / "evaluation_lock.json"
    if lock_path.exists():
        if json.loads(lock_path.read_text(encoding="utf-8")) != lock:
            raise ValueError("Holdout lock changed. Do not retune using holdout or overwrite its results.")
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        lock_path.write_text(json.dumps(lock, indent=2), encoding="utf-8")
    aggregate_dir = output_dir / "aggregate"
    aggregate_dir.mkdir(exist_ok=True)
    results_path = aggregate_dir / "real_holdout_results_5seeds.csv"
    if (output_dir / "evaluation_completed.json").exists():
        print("Real holdout already completed; reading frozen results, not running predictions again.")
        return pd.read_csv(results_path)
    result_rows = []
    for seed in seeds:
        seed_dir = output_dir / f"seed_{seed}"
        completed_path = seed_dir / "evaluation_completed.json"
        if completed_path.exists():
            row = json.loads(completed_path.read_text(encoding="utf-8"))
        else:
            clear_session()
            model = load_model(seed)
            dataset = make_dataset(manifest)
            _ = model.predict(dataset.take(1), verbose=0)  # Warm up tracing.
            start = time.perf_counter()
            scores = np.asarray(model.predict(dataset, verbose=0))
            prediction_seconds = time.perf_counter() - start
            if outputs_are_logits:
                exponentials = np.exp(scores - scores.max(axis=1, keepdims=True))
                scores = exponentials / exponentials.sum(axis=1, keepdims=True)
            row = {"seed": seed, **save_real_holdout_seed(manifest, scores, class_names, seed_dir),
                   "prediction_seconds_including_input_pipeline": prediction_seconds,
                   "inference_time_ms_per_image": 1000.0 * prediction_seconds / len(manifest),
                   "fps_including_input_pipeline": len(manifest) / prediction_seconds}
            completed_path.write_text(json.dumps(row, indent=2), encoding="utf-8")
            del model, scores, dataset
            clear_session()
        result_rows.append(row)
    results = pd.DataFrame(result_rows)
    results.to_csv(results_path, index=False)
    metric_columns = [c for c in results if c not in {"seed", "images", "macro_class_count"}]
    results[metric_columns].agg(["mean", "std"]).transpose().to_csv(aggregate_dir / "real_holdout_mean_std.csv")
    per_class = pd.concat([pd.read_csv(output_dir / f"seed_{seed}" / "per_class_metrics.csv")
                          .assign(seed=seed) for seed in seeds], ignore_index=True)
    per_class.groupby("class")[["precision", "recall", "f1-score"]].agg(["mean", "std"]).to_csv(
        aggregate_dir / "per_class_real_holdout_mean_std.csv")
    deployment_result = next(row for row in result_rows if row["seed"] == deployment_seed)
    (aggregate_dir / "deployment_seed_result.json").write_text(json.dumps({
        "selection_source": "validation only, fixed before holdout", "result": deployment_result,
    }, indent=2), encoding="utf-8")
    (output_dir / "evaluation_completed.json").write_text(json.dumps({
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "note": "No model or seed selection on holdout. SD measures training-seed variability, not sampling uncertainty.",
    }, indent=2), encoding="utf-8")
    print("Real holdout results (macro metrics over supported true classes only):")
    print(results)
    return results


# 19. Guarded, exposed real-world comparison
if RUN_REAL_HOLDOUT:
    assert_selection_lock()
    holdout_archive = resolve_holdout_archive(REAL_HOLDOUT_ARCHIVE_NAME,
                                               REAL_HOLDOUT_ARCHIVE_OVERRIDE)
    holdout_reference_rows = valid_df.assign(
        relative_path=valid_df["path"].map(
            lambda path: Path(path).relative_to(DATASET_PATH).as_posix())
    ).to_dict("records")
    if Path("/content").is_dir():
        holdout_extract_dir = Path("/content/real_holdout_025") / REAL_HOLDOUT_ARCHIVE_SHA256[:12]
    elif Path("/kaggle/working").is_dir():
        holdout_extract_dir = Path("/kaggle/working/real_holdout_025") / REAL_HOLDOUT_ARCHIVE_SHA256[:12]
    else:
        holdout_extract_dir = Path(tempfile.gettempdir()) / "real_holdout_025" / REAL_HOLDOUT_ARCHIVE_SHA256[:12]
    holdout_dir = OUT_DIR / "real_holdout"
    holdout_manifest, holdout_audit = prepare_real_holdout(
        holdout_archive, holdout_extract_dir, CLASS_NAMES,
        REAL_HOLDOUT_ARCHIVE_SHA256, REAL_HOLDOUT_EXPECTED_COUNTS,
        holdout_reference_rows, holdout_dir,
    )
    if len(holdout_manifest) != 120:
        raise ValueError("Holdout05 must have 120 scored images after exact-overlap audit")
    def make_holdout_dataset(manifest):
        dataset = tf.data.Dataset.from_tensor_slices((
            manifest["path"].tolist(), manifest["label"].to_numpy(dtype=np.int32)))
        return dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE).batch(
            BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    def clear_holdout_session():
        keras.backend.clear_session(); gc.collect()
    checkpoint_paths = {seed: SOURCE_DIR / f"seed_{seed}" / "best.weights.h5" for seed in SEEDS}
    holdout_results_df = evaluate_real_holdout(
        holdout_manifest, holdout_audit, CLASS_NAMES, SEEDS, validation_results_df,
        checkpoint_paths, load_selected_model, make_holdout_dataset,
        clear_holdout_session, holdout_dir, outputs_are_logits=False,
    )
    save_json(holdout_dir / "interpretation.json", {
        "status": "EVALUATED_SEPARATELY", "evaluated_images": 120,
        "excluded_exact_test_duplicate": 0,
        "use": "external four-class descriptive evaluation; no Test/holdout tuning",
        "selection_source": "Validation only",
    })
    for normalized, suffix in ((False, ""), (True, "_normalized")):
        fig, axes = plt.subplots(1, len(SEEDS), figsize=(28, 5.5), facecolor="white")
        for column, seed in enumerate(SEEDS):
            matrix = pd.read_csv(holdout_dir / f"seed_{seed}" /
                                 f"confusion_matrix{suffix}.csv", index_col=0).to_numpy()
            ax = axes[column]
            ax.imshow(matrix, cmap=REAL_HOLDOUT_CMAP, vmin=0,
                      vmax=1 if normalized else max(1, matrix.max()))
            ax.set_title(f"Seed {seed}")
            ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
            ax.set_yticks(range(NUM_CLASSES))
            ax.set_yticklabels(CLASS_NAMES if column == 0 else [])
            for row in range(NUM_CLASSES):
                for col in range(NUM_CLASSES):
                    ax.text(col, row, f"{matrix[row, col]:.2f}" if normalized else str(int(matrix[row, col])),
                            ha="center", va="center", color="black", fontsize=8)
            ax.set_xlabel("Predicted")
        axes[0].set_ylabel("True")
        fig.tight_layout()
        fig.savefig(holdout_dir / f"holdout_confusion_matrices{suffix}_5seeds_panel.png",
                    dpi=190, bbox_inches="tight")
        plt.close(fig)
    display(holdout_results_df)
else:
    print("Real holdout PENDING (RUN_REAL_HOLDOUT=False)")


Real holdout: 120 evaluated of 120 archive images; excluded exact Test duplicate: []
Present classes: ['anthracnose', 'healthy', 'leaf_miner', 'red_rust']
Not evaluated (no true examples): ['not_cashew_leaf']
Global seed set to: 42
Global seed set to: 123
Global seed set to: 2026
Global seed set to: 3407
Global seed set to: 7777
Real holdout results (macro metrics over supported true classes only):
   seed  accuracy  macro_precision_present_classes  \
0    42  0.775000                         0.810742   
1   123  0.808333                         0.797431   
2  2026  0.816667                         0.809955   
3  3407  0.800000                         0.795395   
4  7777  0.800000                         0.823983   

   macro_recall_present_classes  macro_f1_present_classes  \
0                      0.731117                  0.757913   
1                      0.744998                  0.764184   
2                      0.756508                  0.773629   
3                      0.7487

,seed,accuracy,macro_precision_present_classes,macro_recall_present_classes,macro_f1_present_classes,balanced_accuracy_present_classes,images,macro_class_count,prediction_seconds_including_input_pipeline,inference_time_ms_per_image,fps_including_input_pipeline
0,42,0.775000,0.810742,0.731117,0.757913,0.731117,120,4,1.352972,11.274767,88.693627
1,123,0.808333,0.797431,0.744998,0.764184,0.744998,120,4,1.347909,11.232577,89.026765
2,2026,0.816667,0.809955,0.756508,0.773629,0.756508,120,4,1.343180,11.193169,89.340208
3,3407,0.800000,0.795395,0.748794,0.767235,0.748794,120,4,1.370929,11.424408,87.531889
4,7777,0.800000,0.823983,0.739563,0.764953,0.739563,120,4,1.352084,11.267369,88.751861


In [14]:
NOTEBOOK_SOURCE_ZLIB_B64 = "..."
# EXP-025 result summary and archive; no checkpoint bytes are copied.
assert_selection_lock()
if not (AGGREGATE_DIR / "test_results_5seeds.csv").is_file():
    raise FileNotFoundError("Final Test results are incomplete")
if not (OUT_DIR / "real_holdout" / "evaluation_completed.json").is_file():
    raise FileNotFoundError("Holdout results are incomplete")
summary = {
    "experiment_id": EXPERIMENT_ID, "source_experiment_id": source_config["experiment_id"],
    "selection_source": "EXP-024 Validation only", "deployment_seed": deployment_seed,
    "training": "NOT_RUN", "final_test_images": EXPECTED_TOTALS["test"],
    "holdout_images": sum(REAL_HOLDOUT_EXPECTED_COUNTS.values()),
    "final_test": "COMPLETE", "holdout": "COMPLETE",
    "source_validation_lock_sha256": file_sha256(source_lock_path),
}
save_json(OUT_DIR / "evaluation_summary.json", summary)
display(pd.read_csv(AGGREGATE_DIR / "test_results_5seeds.csv"))
display(pd.read_csv(OUT_DIR / "real_holdout" / "aggregate" / "real_holdout_results_5seeds.csv"))
notebook_copy = OUT_DIR / "notebook" / "densenet121_eval_024_checkpoints_025.ipynb"
notebook_copy.parent.mkdir(exist_ok=True)
snapshot = json.loads(zlib.decompress(base64.b64decode(NOTEBOOK_SOURCE_ZLIB_B64)))
notebook_copy.write_text(json.dumps(snapshot, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
analysis_zip = RESULTS_ROOT / f"{EXPERIMENT_ID}_ANALYSIS.zip"
with zipfile.ZipFile(analysis_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=f"{EXPERIMENT_ID}/{path.relative_to(OUT_DIR).as_posix()}")
print("EXP-025 evaluation complete:", analysis_zip, "SHA256", file_sha256(analysis_zip))


,seed,images,loss,accuracy,macro_precision,macro_recall,macro_f1,balanced_accuracy,prediction_seconds_including_input_pipeline,inference_time_ms_per_image,real_filename_correct,real_filename_total,real_filename_accuracy,other_filename_correct,other_filename_total,other_filename_accuracy
0,42,685,0.122153,0.957664,0.953562,0.953585,0.953527,0.953585,1.672408,2.441472,26,28,0.928571,630,657,0.958904
1,123,685,0.122853,0.957664,0.954350,0.952975,0.953495,0.952975,1.607844,2.347217,25,28,0.892857,631,657,0.960426
2,2026,685,0.118079,0.960584,0.957527,0.956605,0.956932,0.956605,1.611478,2.352523,25,28,0.892857,633,657,0.963470
3,3407,685,0.111794,0.967883,0.965158,0.964570,0.964818,0.964570,1.615277,2.358068,26,28,0.928571,637,657,0.969559
4,7777,685,0.113768,0.960584,0.957155,0.956337,0.956693,0.956337,1.611637,2.352755,26,28,0.928571,632,657,0.961948


,seed,accuracy,macro_precision_present_classes,macro_recall_present_classes,macro_f1_present_classes,balanced_accuracy_present_classes,images,macro_class_count,prediction_seconds_including_input_pipeline,inference_time_ms_per_image,fps_including_input_pipeline
0,42,0.775000,0.810742,0.731117,0.757913,0.731117,120,4,1.352972,11.274767,88.693627
1,123,0.808333,0.797431,0.744998,0.764184,0.744998,120,4,1.347909,11.232577,89.026765
2,2026,0.816667,0.809955,0.756508,0.773629,0.756508,120,4,1.343180,11.193169,89.340208
3,3407,0.800000,0.795395,0.748794,0.767235,0.748794,120,4,1.370929,11.424408,87.531889
4,7777,0.800000,0.823983,0.739563,0.764953,0.739563,120,4,1.352084,11.267369,88.751861


EXP-025 evaluation complete: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-025-256px_ANALYSIS.zip SHA256 eb2b0e6adcb4594419e09bcf5ea869f89c54d6623b97297e4bac9720e0919fdf


## Interpretation

EXP-025 evaluates 024 checkpoints without retraining or changing the
Validation-selected seed. Final Test has five true classes; Holdout (5) has
four true classes, so compare their reports separately. Holdout has no true
`not_cashew_leaf` support. These metrics describe the frozen 024 models and
must not be used to choose a new checkpoint or resize size.
